# Stage 01 - Data Ingestion: Advanced PDF Extraction and Production Challenges
### Lecture 13: Cleaning Artifacts, Multi-Column Text, and Header/Footer Stripping

This notebook addresses real-world enterprise PDF challenges:
1. Stripping repeating running headers, footers, and page numbers.
2. Normalizing whitespace, line wraps, and hyphenated line-breaks.
3. Production-grade cleaning pipeline for financial and scientific reports.
4. Chunking cleaned text without splitting tabular numbers.


In [1]:
# Step 1: Environment Setup and Raw Extraction
import re
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document

repo_root = Path.cwd()
while not (repo_root / "data").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

pdf_path = repo_root / "data" / "pdf" / "sample_financial_report.pdf"
loader = PyMuPDFLoader(str(pdf_path))
raw_pages = loader.load()

print(f"[INFO] Loaded {len(raw_pages)} raw pages for sanitization.")


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_7192\2121325510.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


[INFO] Loaded 3 raw pages for sanitization.


In [2]:
# Step 2: Advanced Cleaning and Normalization Pipeline
def sanitize_pdf_page_content(text: str) -> str:
    # 1. Remove hyphenated line-breaks (e.g. 'informa- \n tion' -> 'information')
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    # 2. Normalize multiple spaces and non-breaking spaces
    text = re.sub(r'[ \t]+', ' ', text)
    # 3. Collapse multiple newlines to double newlines (paragraphs)
    text = re.sub(r'\n\s*\n+', '\n\n', text)
    # 4. Remove isolated page number footers (e.g., 'Page 1 of 10')
    text = re.sub(r'(?i)page\s+\d+(\s+of\s+\d+)?', '', text)
    return text.strip()

cleaned_documents = []
for page in raw_pages:
    clean_text = sanitize_pdf_page_content(page.page_content)
    if clean_text:
        cleaned_documents.append(
            Document(
                page_content=clean_text,
                metadata={**page.metadata, "is_sanitized": True}
            )
        )

print(f"[INFO] Sanitized {len(cleaned_documents)} pages.")
print(f"Sample Sanitized Content:\n{cleaned_documents[0].page_content[:200]}...")

# Assertions
assert len(cleaned_documents) > 0, "Sanitization must yield non-empty documents."
assert all(d.metadata["is_sanitized"] is True for d in cleaned_documents), "Metadata must flag sanitization."
print("[SUCCESS] Production PDF sanitization pipeline verified.")


[INFO] Sanitized 3 pages.
Sample Sanitized Content:
Company Financial Report 2024
The financial performance for fiscal year 2024 shows significant growth in profitability.
Revenue increased by 25% due to efficient operations and new product launches.
O...
[SUCCESS] Production PDF sanitization pipeline verified.
